In [44]:
!pip install docling "docling[asr]" semantic-chunking pymupdf requests pillow transformers python-docx python-pptx python-magic

In [64]:

from docling.document_converter import DocumentConverter, AudioFormatOption
from docling.datamodel import asr_model_specs
from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import AsrPipelineOptions
from docling.pipeline.asr_pipeline import AsrPipeline
from semantic_chunking import SemanticChunker
import requests
from io import BytesIO
import fitz
from PIL import Image
from transformers import BlipProcessor, BlipForConditionalGeneration
from docx import Document
from pptx import Presentation
import magic

# function for generating captions for image
def imageCaptioning(link):

  resp = requests.get(link)
  file_stream = BytesIO(resp.content)
  mime = magic.from_buffer(file_stream.getvalue(), mime=True)

  images = []

  if mime == "application/pdf":
    docs = fitz.open(stream=file_stream, filetype="pdf")

    for page in docs:
      for img in page.get_images(full=True):
        xref = img[0]
        base_image = docs.extract_image(xref)
        image_bytes = base_image["image"]
        image = Image.open(BytesIO(image_bytes)).convert("RGB")
        images.append(image)

  elif mime == "application/vnd.openxmlformats-officedocument.wordprocessingml.document":
    doc = Document(file_stream)

    for rel in doc.part.rels.values():
      if "image" in rel.target_ref:
        image_bytes = rel.target_part.blob
        image = Image.open(BytesIO(image_bytes)).convert("RGB")
        images.append(image)

  elif mime == "application/vnd.ms-powerpoint":
    prs = Presentation(file_stream)

    for slide in prs.slides:
      for shape in slide.shapes:
        if shape.shape_type == 13: # picture
          image_bytes = shape.image.blob
          image = Image.open(BytesIO(image_bytes)).convert("RGB")
          images.append(image)

  processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
  model = BlipForConditionalGeneration.from_pretrained("Salesforce/blip-image-captioning-base")

  captions = []

  for img in images:
    inputs = processor(img, return_tensors="pt")
    out = model.generate(**inputs)
    caption = processor.decode(out[0], skip_special_tokens=True)
    captions.append(caption)

  return captions

# function for semantic chunking of texts
def semanticChunking(text):
  chunker = SemanticChunker(model_name='all-MiniLM-L6-v2', max_chunk_size=1500, similarity_threshold=0.7)
  chunks = chunker.semantic_chunk(text)

  print(f"Total chunk length : {len(chunks)}")
  print("Semantic Chunks:")
  for chunk in chunks:
    print(f"{chunk}\n")

# function for document extraction
def extractDocument(link):
  pdfconverter = DocumentConverter()
  doc = pdfconverter.convert(link).document
  content = doc.export_to_markdown()
  print(content)
  captions = imageCaptioning(link)

  if len(captions) > 0:
    content += "\n\n IMAGE DESCRIPTIONS : \n\n"
    for text in captions:
      content += f"{text}\n"
  print(content)
  # semanticChunking(content)

# function for audio & video extraction
def mediaExtraction(link):
  pipeline_options = AsrPipelineOptions()
  pipeline_options.asr_options = asr_model_specs.WHISPER_TURBO

  mediaconverter = DocumentConverter(
       format_options = {
          InputFormat.AUDIO: AudioFormatOption(
             pipeline_cls = AsrPipeline,
             pipeline_options = pipeline_options
         )
      }
  )

  result = mediaconverter.convert(link).document
  content = result.export_to_markdown()
  print(content)
  # semanticChunking(content)

link = input("Enter a link : ")

if "google.com" in link:
  print("URL not allowed")
else:
  resp = requests.get(link)
  file_stream = BytesIO(resp.content)
  mime = magic.from_buffer(file_stream.getvalue(), mime=True)

  if mime == "audio/mpeg" or mime == "video/mp4":
    mediaExtraction(link)
  else:
    extractDocument(link)

Enter a link : https://lfjakrpqbidggevkatwm.supabase.co/storage/v1/object/public/temp/Post%20PHD%20P2.pdf


[INFO] 2026-04-07 15:15:21,687 [RapidOCR] base.py:22: Using engine_name: torch
[INFO] 2026-04-07 15:15:21,689 [RapidOCR] device_config.py:57: Using CPU device
[INFO] 2026-04-07 15:15:21,738 [RapidOCR] download_file.py:60: File exists and is valid: /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_PP-OCRv4_det_infer.pth
[INFO] 2026-04-07 15:15:21,739 [RapidOCR] main.py:50: Using /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_PP-OCRv4_det_infer.pth
[INFO] 2026-04-07 15:15:21,987 [RapidOCR] base.py:22: Using engine_name: torch
[INFO] 2026-04-07 15:15:21,988 [RapidOCR] device_config.py:57: Using CPU device
[INFO] 2026-04-07 15:15:21,993 [RapidOCR] download_file.py:60: File exists and is valid: /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_ptocr_mobile_v2.0_cls_infer.pth
[INFO] 2026-04-07 15:15:21,994 [RapidOCR] main.py:50: Using /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_ptocr_mobile_v2.0_cls_infer.pth
[INFO] 2026-04-07 15:15:22,091 [RapidO

Loading weights:   0%|          | 0/770 [00:00<?, ?it/s]

## A Correlative Survey on Imbalanced Breast Cancer Data Using Ensembled Oversampling Techniques and Deep Learning Algorithms

*Debaleena Datta Department of CSE Techno India Saltlake Kolkata, India leenadatta1907@gmail.com

Pradeep Kumar Mallick Department of CSE Kalinga Institute of Industrial Technology, Bhubaneswar, India

pradeep.mallickfcs@kiit.ac.in

Abstract -Inadequacy of labeled breast cancer data causes imbalance and inaccuracy in classifiers. Our proposed research aims for a computer assisted detection (CAD) system to detect maligned cancer cells using the Wisconsin Breast Cancer dataset through a novel ensembled approach that combines various oversampling methods with standard machine learning and deep learning classifiers. The work is three-folds: (1) Use of 6 oversampling techniques: Random oversampling, Synthetic minority oversampling technique, Borderline SMOTE, k-means SMOTE, Support vector machine SMOTE, Adaptive synthetic minority oversampling, (2) Use of 5 ML model

Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

BlipForConditionalGeneration LOAD REPORT from: Salesforce/blip-image-captioning-base
Key                                       | Status     |  | 
------------------------------------------+------------+--+-
text_decoder.bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
/usr/local/lib/python3.12/dist-packages/transformers/generation/utils.py:1569: UserWarning: Using the model-agnostic default `max_length` (=21) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(


## A Correlative Survey on Imbalanced Breast Cancer Data Using Ensembled Oversampling Techniques and Deep Learning Algorithms

*Debaleena Datta Department of CSE Techno India Saltlake Kolkata, India leenadatta1907@gmail.com

Pradeep Kumar Mallick Department of CSE Kalinga Institute of Industrial Technology, Bhubaneswar, India

pradeep.mallickfcs@kiit.ac.in

Abstract -Inadequacy of labeled breast cancer data causes imbalance and inaccuracy in classifiers. Our proposed research aims for a computer assisted detection (CAD) system to detect maligned cancer cells using the Wisconsin Breast Cancer dataset through a novel ensembled approach that combines various oversampling methods with standard machine learning and deep learning classifiers. The work is three-folds: (1) Use of 6 oversampling techniques: Random oversampling, Synthetic minority oversampling technique, Borderline SMOTE, k-means SMOTE, Support vector machine SMOTE, Adaptive synthetic minority oversampling, (2) Use of 5 ML model

In [61]:

import magic
import requests
from io import BytesIO

link = input("Enter your URL : ")

if "google.com" in link:
  print("URL not allowed")
else:
  resp = requests.get(link)
  file_stream = BytesIO(resp.content)

  mime = magic.from_buffer(file_stream.getvalue(), mime=True)
  print(mime)

Enter your URL : https://docs.google.com/presentation/d/1DXAFR5q7nzequ5jGK1TBSq-cRlW-AJk4/edit?usp=drivesdk&ouid=114511759470220354113&rtpof=true&sd=true
URL not allowed
